# exp08 - four gaps closed (Colab, multi-session)

Four gaps flagged in a code review of `core/configs.py`, none needing a new circuit
template - just crossings of knobs that already exist. Full rationale in
`core/configs.py`'s `[exp08 follow-up]` comments and `docs/CORRECTIONS.md#new-08`.

| tier | gap | arms | writes to |
|---|---|---|---|
| 0 | Fourier ceiling never covered FrozenLake Config A | `frozen_scalar_1q_fourier_ceiling_L{1,5,10,15}` (4) | `exp04`'s directory |
| 1 | Hsiao OR only at n=3 (exp07's own coverage pass) | `paper_hsiao_or_r{4,8,16,32}` + `_r{4,16}_ent` (6) | `exp07`'s directory (continues it) |
| 2 | DR x OR never crossed, L=1 | `hybrid_DR1_OR{4,8,16,32}` (4) | `exp08`'s own directory |
| 3 | DR x OR never crossed, L=2 | `hybrid_DR2_OR{4,8,16,32}` (4) | `exp08`'s own directory |
| 4 | Entanglement never ablated on exp01/exp02's own arms | `hybrid_fig4_noent`, `hybrid_or_r{4,8,16,32}_noent` (5) | `exp08`'s own directory |

Ordered cheapest -> most expensive, same reasoning as `12_cartpole_paper_replication_colab.ipynb`:
qubit count dominates cost, then depth. Tier 4 (8 qubits, L=5) is the expensive one.

## Multi-session seeds

**This notebook is meant to run in up to 4 Colab sessions AT ONCE.** Section 3 has a
single `SEEDS` list used by every tier - set it differently in each session before
running anything else, e.g.:

| session | SEEDS |
|---|---|
| A | `[1, 2, 3]` |
| B | `[4, 5, 6]` |
| C | `[7, 8]` |
| D | `[9, 10]` |

Every cell is checkpointed by its own manifest AND cooperatively locked
(`claim=True`, `dqn/runner.py::claim_cell`) for the duration of training, so two
sessions racing on the same seed by mistake do not double-train it - the second one
skips with `busy - another session holds this cell` instead. Tier 1 already has
seeds 1-3 on disk from `12_cartpole_paper_replication_colab.ipynb` - a session that
includes 1-3 in its `SEEDS` just gets an instant reuse hit on those, no harm.

## Progress, without the endless step-by-step log

Training normally prints one line per EPISODE (`global_step=..., episodic_return=...`)
- upstream's own behaviour (`simplyqrl/dqn.py`), not something this project's code
generates. Passing `progress_bar=True` (this notebook does, throughout) switches
that off and shows a single, in-place `tqdm` bar per cell instead - one line that
updates, not thousands that scroll. Between cells you still get one summary line
(`[i/N] name` then `ok <seconds> phantoms <pct> FIX-01 <on/off>`) - so the whole
notebook's transcript stays short regardless of how many steps run.

Section 6 gives a clean, compute-free progress table across ALL FOUR sessions at
once (they all write to the same Drive folders) - run it any time, in any session,
without waiting for training to reach a stopping point.

---
## 1. Environment

In [ ]:
import os, sys, subprocess, pathlib

GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = (f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok
                else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    CODE    = pathlib.Path("/content/qrl-dissection")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/results")
else:
    CODE    = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    RESULTS = pathlib.Path.cwd() / "results"
RES = RESULTS  # alias, see notebooks/README.md

if IN_COLAB:
    if not CODE.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(CODE)], check=True)
    else:
        subprocess.run(["git", "-C", str(CODE), "pull"], check=True)
    # SimplyQRL is vendored in this repo, so `pip install -e .` is the whole
    # install - see docs/CORRECTIONS.md#fix-04.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    # Colab preinstalls jax, and PennyLane imports it opportunistically if it
    # is present - it is absent from both the upstream lock and this repo's
    # requirements.txt (see that file's own note). Uninstall it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

sys.path.insert(0, str(CODE / "src"))
assert CODE.exists() and RESULTS is not None

# If jax (or the broken autoray shim FIX-04 documents) is already imported in
# THIS kernel - e.g. Colab's own startup, or a previous run of this cell -
# uninstalling the package does not unload it from memory. Only a runtime
# restart does.
_need_restart = "jax" in sys.modules
if "autoray.autoray" in sys.modules:
    import autoray.autoray as _aa
    _need_restart = _need_restart or not hasattr(_aa, "NumpyMimic")
if _need_restart:
    print("Incompatible modules already loaded -> restarting runtime.")
    os.kill(os.getpid(), 9)

OUT_FL = RESULTS / "exp04_dqn_frozenlake_embeddings"          # tier 0 (reuse target)
OUT_P7 = RESULTS / "exp07_dqn_cartpole_paper_replication"      # tier 1 (continues exp07)
OUT_08 = RESULTS / "exp08_dqn_cartpole_frozenlake_followups"   # tiers 2-4 (new)
for d in (OUT_FL, OUT_P7, OUT_08):
    d.mkdir(parents=True, exist_ok=True)
print("CODE   =", CODE)
print("OUT_FL =", OUT_FL)
print("OUT_P7 =", OUT_P7)
print("OUT_08 =", OUT_08)

---
## 2. Preflight

Same gate as the other runner notebooks - the full test suite, before spending any
compute. Skip only if you already ran it this session.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
                   cwd=str(CODE), capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0, "test suite failing - do not run the grid against it"
print("\npreflight OK")

---
## 3. SESSION CONFIG - set this differently in each of the 4 sessions

Edit `SEEDS` below BEFORE running anything past this cell. Suggested 10-seed split
across 4 sessions: `[1,2,3]` / `[4,5,6]` / `[7,8]` / `[9,10]` - adjust freely, the
only requirement is that what you type here is what THIS session trains.

In [ ]:
SEEDS = [1, 2, 3]   # <-- EDIT PER SESSION, see the table in section 3 above

print(f"This session will train seeds: {SEEDS}")

---
## 4. Shared setup

Same `DQN_KWARGS`/`steps=100_000` as exp02/exp03/exp07 (all CartPole block
sweeps), `fix_autoreset=True` only. `claim=True` + a 15h TTL (comfortably above
the ~9-11h worst-case cell measured on this project's own 8-qubit L=5 arms) is
what makes 4 simultaneous sessions safe - a cell already claimed by another
session is skipped with `busy`, not retrained.

In [ ]:
import qrl_dissection
from qrl_dissection.dqn import GreedyEvalConfig, RunSpec, run_grid
from qrl_dissection.core.obs_adapters import FROZEN_SCALAR_ID

print(qrl_dissection.upstream_report())

DQN_KWARGS = {"batch_size": 128, "buffer_size": 10_000, "train_frequency": 10}
STEPS = 100_000
EVAL_EVERY = 10_000
CP_ENV = "CartPole-v1"
FL_ENV = FROZEN_SCALAR_ID
CP_EVAL_CFG = GreedyEvalConfig(env_id=CP_ENV, every_steps=EVAL_EVERY, n_episodes=20)
FL_EVAL_CFG = GreedyEvalConfig(env_id=FL_ENV, every_steps=EVAL_EVERY, n_episodes=20)

def run_tier(name, arms, outdir, env_id=CP_ENV, eval_cfg=None):
    eval_cfg = eval_cfg or (FL_EVAL_CFG if env_id == FL_ENV else CP_EVAL_CFG)
    specs = [RunSpec(arm=a, seed=s, fix_autoreset=True, total_timesteps=STEPS,
                     dqn_kwargs=DQN_KWARGS)
             for a in arms for s in SEEDS]
    print(f"\n=== {name}: {len(arms)} arms x {len(SEEDS)} seeds = {len(specs)} cells "
          f"(SEEDS={SEEDS}) ===")
    return run_grid(specs, outdir, env_id=env_id, eval_cfg=eval_cfg,
                     progress_bar=True, claim=True, claim_ttl_hours=15.0)

### Tier 0 - FrozenLake Config A Fourier ceiling (classical, near-free)

Bounds each depth in `frozen_scalar_1q_L{1,5,10,15}` with the SAME class the real
circuit could express if unentangled - GAP 3. Writes into exp04's own directory.

In [ ]:
TIER0 = [f"frozen_scalar_1q_fourier_ceiling_L{L}" for L in (1, 5, 10, 15)]
_ = run_tier("Tier 0 - FrozenLake Config A Fourier ceiling", TIER0, OUT_FL, env_id=FL_ENV)

### Tier 1 - Hsiao Output-Reuse, topping up n=3 -> n=10 (GAP 4)

Already has seeds 1-3 on disk (`12_cartpole_paper_replication_colab.ipynb`) - if
`SEEDS` includes any of those, they are instant reuse hits, not retrained.

In [ ]:
TIER1 = ([f"paper_hsiao_or_r{R}" for R in (4, 8, 16, 32)]
          + [f"paper_hsiao_or_r{R}_ent" for R in (4, 16)])
_ = run_tier("Tier 1 - Hsiao OR top-up", TIER1, OUT_P7)

### Tier 2 - DR x OR crossing, L=1 (GAP 2, cheap half)

`hybrid_DR{1,2}` (exp03) sweep depth with OR off; `hybrid_or_r{R}` (exp02) sweep R
at L=5. Neither crosses the other. This is the L=1 half of that crossing.

In [ ]:
TIER2 = [f"hybrid_DR1_OR{R}" for R in (4, 8, 16, 32)]
_ = run_tier("Tier 2 - DR x OR, L=1", TIER2, OUT_08)

### Tier 3 - DR x OR crossing, L=2 (GAP 2, deeper half)

In [ ]:
TIER3 = [f"hybrid_DR2_OR{R}" for R in (4, 8, 16, 32)]
_ = run_tier("Tier 3 - DR x OR, L=2", TIER3, OUT_08)

### Tier 4 - entanglement ablation on exp01/exp02's OWN arms (GAP 1, the expensive one)

8 qubits, L=5 - the same cost class as `hybrid_fig4` itself (measured ~8-9.5h/seed
on this project's own hardware). If a session is going to run out of time, this is
where it happens - everything above will already be banked.

In [ ]:
TIER4 = ["hybrid_fig4_noent"] + [f"hybrid_or_r{R}_noent" for R in (4, 8, 16, 32)]
_ = run_tier("Tier 4 - entanglement ablation (8q, L=5)", TIER4, OUT_08)

---
## 5. Resuming

Re-running section 4's cells (with whatever `SEEDS` this session is assigned) costs
nothing for what already has a manifest, in this session or any of the other 3.

---
## 6. Progress across ALL sessions - compute-free, run any time

Reads every manifest in the three output directories directly - no training, safe
to run from any of the 4 sessions, or a 5th session just to check.

In [ ]:
import json

def progress(outdir, arms):
    print(f"\n--- {outdir.name} ---")
    for arm in arms:
        seeds = set()
        for mp in outdir.glob(f"{arm}__fix01on__s*.manifest.json"):
            m = json.loads(mp.read_text())
            s = m.get("spec", {}).get("seed")
            if s is not None:
                seeds.add(s)
        print(f"  {arm:35s} n={len(seeds):2d}  seeds={sorted(seeds)}")

progress(OUT_FL, TIER0)
progress(OUT_P7, TIER1)
progress(OUT_08, TIER2 + TIER3 + TIER4)